# Circuit Analysis and Electrodynamics: Equations, Variables, and Derivation

This notebook follows a textbook flow:
**field laws (Maxwell) -> constitutive relations -> circuit reduction -> Lagrangian/Hamiltonian form -> data-facing surrogate**.

Renderer note: Mermaid blocks depend on notebook renderer support. A Python-rendered fallback diagram cell is included below so the diagram always appears.

## Model Diagram (Field to Circuit to Data)
Mermaid source is kept for portability, but many notebook renderers do not embed Mermaid directly.
Run **Cell 2** to view the guaranteed rendered theory diagram.

## Full Maxwell System Used for Circuit Interpretation
For electrodynamics in matter:
1. $\nabla\cdot\mathbf{D}=\rho_f$.
2. $\nabla\cdot\mathbf{B}=0$.
3. $\nabla\times\mathbf{E}=-\partial\mathbf{B}/\partial t$.
4. $\nabla\times\mathbf{H}=\mathbf{J}_f+\partial\mathbf{D}/\partial t$.
5. Constitutive closure: $\mathbf{D}=\epsilon\mathbf{E}$, $\mathbf{B}=\mu\mathbf{H}$, $\mathbf{J}_f=\sigma\mathbf{E}$.

## Independent and Dependent Variables (and Why)
- Independent variables:
  - $t$ (time): required for dynamics and transients.
  - $(x,y,z)$ (space): required because Maxwell equations are field PDEs.
- Dependent variables:
  - Field level: $\mathbf{E}(x,y,z,t),\mathbf{H}(x,y,z,t),\mathbf{D}(x,y,z,t),\mathbf{B}(x,y,z,t)$.
  - Reduced circuit level: charge $q(t)$, current $i(t)=\dot q(t)$, flux linkage $\phi(t)$.
- Why this split:
  - Fields describe where energy is stored and transported.
  - Lumped variables are the low-dimensional projection used for fitting and control.

## Trivial Case Before Full Problem
We solve the lossless single-mode LC limit first:
1. $L\ddot q + \frac{1}{C}q=0$.
2. $\omega_0=1/\sqrt{LC}$.
3. $q(t)=Q_0\cos(\omega_0 t)$, $i(t)=\dot q(t)=-Q_0\omega_0\sin(\omega_0 t)$.
4. Energy check:
   $$
   \mathcal{E}(t)=\frac{q(t)^2}{2C}+\frac{L i(t)^2}{2}=\frac{Q_0^2}{2C}=\text{constant}.
   $$

The next code cell renders this trivial case both mathematically (printed identities) and graphically (time-series + energy conservation plot).

In [ ]:
# Theory diagram renderer (always visible in notebooks, independent of Mermaid support).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
blocks = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Model Evaluation / Data Fit'),
]
for x0, y0, label in blocks:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#eef4ff')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Theory Diagram (Rendered Fallback)')
plt.tight_layout()
plt.show()

## Mechanics and Electrodynamics Bridge

### From Maxwell PDEs to a Lumped ODE
1. Start from Faraday and Ampere-Maxwell in integral form over device geometry.
2. Use mode/geometry assumptions to define a dominant generalized coordinate $q(t)$.
3. Obtain effective parameters $L$ and $C$ from magnetic/electric field energy.
4. Reduced equation: $L\ddot q + R\dot q + \frac{1}{C}q = v_s(t)$.

### Variational Form (Lossless Core)
1. Choose generalized coordinate $q$.
2. $T_{\text{mag}}=\frac{1}{2}L\dot q^2$.
3. $V_{\text{elec}}=\frac{1}{2C}q^2$.
4. Lagrangian: $\mathcal{L}(q,\dot q)=T_{\text{mag}}-V_{\text{elec}}$.
5. Euler-Lagrange gives $L\ddot q + \frac{1}{C}q=0$.

### Hamiltonian Form
1. Canonical momentum: $p=\partial\mathcal{L}/\partial\dot q=L\dot q$.
2. Hamiltonian:
   $$
   \mathcal{H}(q,p)=\frac{p^2}{2L}+\frac{q^2}{2C}.
   $$
3. Canonical equations recover the same oscillator dynamics.

### Data-Facing Variable Roles
1. Independent variable in fit cell: normalized time-like index $x\in[0,1]$.
2. Dependent measured variable: scalar observable $y(x)$ (voltage/current/proxy column from dataset).
3. Fitted parameters $(A,\gamma,\omega,c_0)$ map to amplitude, damping, frequency, and offset.
4. This surrogate is an observable approximation to the mechanistic state model above.

## Public Data Sources and Peer-Reviewed References

1. **UCI Electrical Grid Stability**
   - URL: https://archive.ics.uci.edu/ml/machine-learning-databases/00471/Data_for_UCI_named.csv
   - Peer-reviewed context: Arzamasov et al., Int. J. Electr. Power Energy Syst. (2018)
2. **Stock Data (CSV fallback)**
   - URL: https://raw.githubusercontent.com/plotly/datasets/master/stockdata.csv
   - Peer-reviewed context: Public benchmark fallback

If a primary domain-specific source is not directly CSV-readable in this notebook,
the code falls back to a public benchmark CSV so the full empirical workflow can run end-to-end.


In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import requests
import ipywidgets as widgets
from ipywidgets import interact

plt.style.use('seaborn-v0_8-whitegrid')

DOMAIN_KEY = '06_circuit_analysis'
DOMAIN_TITLE = 'Circuit Analysis and EM Fields'
MODEL_FAMILY = 'damped'
DATA_SOURCES = [
  {
    "name": "UCI Electrical Grid Stability",
    "url": "https://archive.ics.uci.edu/ml/machine-learning-databases/00471/Data_for_UCI_named.csv",
    "reference": "Arzamasov et al., Int. J. Electr. Power Energy Syst. (2018)"
  },
  {
    "name": "Stock Data (CSV fallback)",
    "url": "https://raw.githubusercontent.com/plotly/datasets/master/stockdata.csv",
    "reference": "Public benchmark fallback"
  }
]


In [ ]:
# Fallback model diagram renderer (works even when Mermaid is not embedded by the notebook renderer).
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axis('off')
nodes = [
    (0.04, 0.68, 'Physical Assumptions'),
    (0.04, 0.40, 'Lagrangian / Governing PDE'),
    (0.04, 0.12, 'Reduced Dynamic Model'),
    (0.56, 0.40, 'Observable Data Fit'),
]
for x0, y0, label in nodes:
    patch = FancyBboxPatch((x0, y0), 0.36, 0.18, boxstyle='round,pad=0.02', linewidth=1.2, facecolor='#edf3fb')
    ax.add_patch(patch)
    ax.text(x0 + 0.18, y0 + 0.09, label, ha='center', va='center', fontsize=10)
ax.annotate('', xy=(0.22, 0.62), xytext=(0.22, 0.50), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.22, 0.34), xytext=(0.22, 0.22), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.annotate('', xy=(0.56, 0.49), xytext=(0.40, 0.21), arrowprops=dict(arrowstyle='->', lw=1.3))
ax.set_title('Fallback Diagram (Python-rendered)')
plt.tight_layout()
plt.show()

In [ ]:
# Trivial-case diagnostics for a lossless LC oscillator and field-to-circuit map.
from matplotlib.patches import FancyBboxPatch

L0 = 1.2
C0 = 0.8
Q0 = 1.0
omega0 = 1.0 / np.sqrt(L0 * C0)

t_demo = np.linspace(0.0, 25.0, 900)
q_demo = Q0 * np.cos(omega0 * t_demo)
i_demo = -Q0 * omega0 * np.sin(omega0 * t_demo)
energy_e = q_demo**2 / (2.0 * C0)
energy_m = 0.5 * L0 * i_demo**2
energy_total = energy_e + energy_m

print("Trivial LC case:")
print(f"L = {L0:.3f}, C = {C0:.3f}, omega0 = 1/sqrt(LC) = {omega0:.5f}")
print("q(t) = Q0 cos(omega0 t), i(t) = -Q0 omega0 sin(omega0 t)")
print(f"Energy drift (max-min): {np.max(energy_total) - np.min(energy_total):.3e}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# Simple diagram that always renders even when Mermaid is unavailable.
ax0 = ax[0]
ax0.axis("off")
boxes = [
    (0.05, 0.72, "Maxwell PDEs"),
    (0.05, 0.46, "Constitutive Laws"),
    (0.05, 0.20, "Lumped State q,i"),
    (0.58, 0.46, "Data Observable y(t)"),
]
for x0, y0, txt in boxes:
    patch = FancyBboxPatch((x0, y0), 0.35, 0.18, boxstyle="round,pad=0.02", linewidth=1.2, facecolor="#eef3ff")
    ax0.add_patch(patch)
    ax0.text(x0 + 0.175, y0 + 0.09, txt, ha="center", va="center", fontsize=10)
ax0.annotate("", xy=(0.225, 0.64), xytext=(0.225, 0.54), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.225, 0.38), xytext=(0.225, 0.28), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.annotate("", xy=(0.58, 0.55), xytext=(0.40, 0.29), arrowprops=dict(arrowstyle="->", lw=1.3))
ax0.set_title("Model Diagram")

ax[1].plot(t_demo, q_demo, label="q(t): charge")
ax[1].plot(t_demo, i_demo, label="i(t): current")
ax[1].set_title("Trivial LC State Trajectories")
ax[1].set_xlabel("time t")
ax[1].set_ylabel("state value")
ax[1].legend()

ax[2].plot(t_demo, energy_e, label="electric energy q^2/(2C)")
ax[2].plot(t_demo, energy_m, label="magnetic energy L i^2/2")
ax[2].plot(t_demo, energy_total, "k--", linewidth=2.0, label="total")
ax[2].set_title("Energy Conservation (Lossless)")
ax[2].set_xlabel("time t")
ax[2].set_ylabel("energy")
ax[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
def try_read_public_csv(url: str):
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        if 'text/csv' in r.headers.get('Content-Type', '') or ',' in r.text[:2000]:
            return pd.read_csv(io.StringIO(r.text))
    except Exception:
        return None
    return None

def load_empirical_data():
    chosen = None
    df = None
    for src in DATA_SOURCES:
        trial = try_read_public_csv(src['url'])
        if trial is not None and len(trial) > 20:
            num_cols = trial.select_dtypes(include=[np.number]).columns.tolist()
            if len(num_cols) >= 1:
                df = trial.copy()
                chosen = src
                break

    if df is None:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (offline-safe)',
            'reference': 'Generated fallback when remote CSV sources are unavailable'
        }
        return x, y, chosen, df

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) == 1:
        x = np.arange(len(df), dtype=float)
        y = df[numeric_cols[0]].to_numpy(dtype=float)
    else:
        x = df[numeric_cols[0]].to_numpy(dtype=float)
        y = df[numeric_cols[1]].to_numpy(dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if x.size < 8 or y.size < 8:
        x = np.linspace(0.0, 1.0, 600)
        y = np.exp(-2.5 * x) * np.cos(10.0 * x) + 0.05 * np.sin(40.0 * x)
        df = pd.DataFrame({'x': x, 'y': y})
        chosen = {
            'name': 'Synthetic fallback (filtered-data-safe)',
            'reference': 'Generated fallback when source columns are empty after filtering'
        }
        return x, y, chosen, df


    if len(x) > 2000:
        step = max(1, len(x) // 2000)
        x = x[::step]
        y = y[::step]

    if np.ptp(x) == 0:
        x = np.arange(len(x), dtype=float)

    x = (x - np.min(x)) / max(np.ptp(x), 1e-12)
    return x, y, chosen, df

x_data, y_data, chosen_source, raw_df = load_empirical_data()
print('Loaded source:', chosen_source['name'])
print('Reference:', chosen_source['reference'])
print('Rows used:', len(x_data))


In [ ]:
def model_function(x, p1, p2, p3, p4):
    if MODEL_FAMILY == 'oscillator':
        A, gamma, omega, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) * np.cos(omega * x) + c
    if MODEL_FAMILY == 'damped':
        A, gamma, _, c = p1, p2, p3, p4
        return A * np.exp(-gamma * x) + c
    if MODEL_FAMILY == 'logistic':
        K, r, x0, c = p1, p2, p3, p4
        return K / (1.0 + np.exp(-r * (x - x0))) + c
    if MODEL_FAMILY == 'michaelis':
        vmax, km, _, c = p1, p2, p3, p4
        return vmax * x / (km + x + 1e-12) + c
    if MODEL_FAMILY == 'diffusion':
        A, D, _, c = p1, p2, p3, p4
        return A * np.exp(-D * x) + c
    return p1 * x + p2

p0 = [np.nanmax(y_data) - np.nanmin(y_data), 0.5, 6.0, np.nanmean(y_data)]
bounds = ([-1e6, 0.0, -1e3, -1e6], [1e6, 50.0, 1e3, 1e6])

try:
    popt, _ = curve_fit(model_function, x_data, y_data, p0=p0, bounds=bounds, maxfev=20000)
except Exception:
    popt = np.array(p0, dtype=float)

y_hat = model_function(x_data, *popt)
rmse = float(np.sqrt(np.mean((y_data - y_hat) ** 2)))
r2 = float(1.0 - np.sum((y_data - y_hat) ** 2) / max(np.sum((y_data - np.mean(y_data)) ** 2), 1e-12))

print('Fitted parameters:', popt)
print('RMSE:', rmse)
print('R^2:', r2)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(x_data, y_data, s=12, alpha=0.5, label='Empirical data')
ax.plot(x_data, y_hat, color='crimson', linewidth=2.0, label='Fitted surrogate model')
ax.set_title(f'{DOMAIN_TITLE}: empirical fit ({MODEL_FAMILY})')
ax.set_xlabel('Normalized independent variable')
ax.set_ylabel('Observed variable')
ax.legend()
plt.show()


## Interpreting Model Evaluation Trends and Values

Use this guide when reading fitted parameters, RMSE, and $R^2$ outputs:

1. **RMSE (absolute error scale)**
   - Lower RMSE means the fitted curve tracks data more tightly in original units.
   - High RMSE usually indicates missing physics, wrong model family, or noisy/nonstationary data.

2. **$R^2$ (explained variance)**
   - Near 1.0: surrogate captures dominant trend/oscillation structure.
   - Near 0.0: surrogate explains little beyond the mean level.
   - Negative: model is worse than using a constant mean prediction.

3. **Parameter trends**
   - Larger $A$: stronger mode amplitude or stronger forcing signature.
   - Larger $\gamma$: faster decay / stronger damping / shorter memory.
   - Larger $\omega$: faster oscillation or higher characteristic frequency.
   - Offset $c_0$: baseline bias or slowly varying background level.

4. **Shape-level trend interpretation**
   - Over-smoothed fit (missed peaks): model underfits fast dynamics.
   - Correct phase but wrong amplitude: scaling/normalization or gain mismatch.
   - Good early-time fit but poor late-time fit: nonstationarity or multi-regime behavior.

5. **Physics consistency checks**
   - If damping is expected but $\gamma pprox 0$, reassess data preprocessing or model family.
   - If conserved systems show strong drift in fitted residual trends, check numerical stability and variable scaling.
   - Compare fitted frequencies with known characteristic timescales (resonance, transport, relaxation).

## Interactive Simulation Controls

Use the sliders below to perturb model parameters and observe how well the
simulation matches the empirical measurements.


In [ ]:
def interactive_plot(p1, p2, p3, p4):
    y_sim = model_function(x_data, p1, p2, p3, p4)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.scatter(x_data, y_data, s=10, alpha=0.45, label='Empirical data')
    ax.plot(x_data, y_sim, color='navy', linewidth=2.0, label='Simulation')
    ax.set_title(f'{DOMAIN_TITLE}: interactive parameter sweep')
    ax.set_xlabel('Normalized independent variable')
    ax.set_ylabel('Observed variable')
    ax.legend()
    plt.show()

interact(
    interactive_plot,
    p1=widgets.FloatSlider(value=float(popt[0]), min=-2*abs(float(popt[0]))-1, max=2*abs(float(popt[0]))+1, step=0.01),
    p2=widgets.FloatSlider(value=float(popt[1]), min=0.0, max=max(1.0, 3*abs(float(popt[1]))+1), step=0.01),
    p3=widgets.FloatSlider(value=float(popt[2]), min=-20.0, max=20.0, step=0.05),
    p4=widgets.FloatSlider(value=float(popt[3]), min=float(np.min(y_data))-2, max=float(np.max(y_data))+2, step=0.01),
);
